# **Exploration & Analyse de données - Préparation**

## **Imports**

In [ ]:
# Import des librairies nécessaires
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import random
import warnings
warnings.filterwarnings('ignore')

# 
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

In [ ]:
# Chargement du dataset
df = pd.read_csv('../data/medquad.csv')

In [ ]:
# Dimensions
print(f"Shape: {df.shape}")
print(f"Lignes: {df.shape[0]}, Colonnes: {df.shape[1]}")
print("\nColonnes:", df.columns.tolist())

# Info générale
print("\nInfo dataset:")
df.info()

# Aperçu premières lignes
print("\nAperçu:")
df.head()

In [ ]:
# Valeurs manquantes
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({'Manquants': missing, '%': missing_pct})
print("Valeurs manquantes:")
print(missing_df[missing_df['Manquants'] > 0])

In [ ]:
# Doublons
print(f"\nDoublons exacts: {df.duplicated().sum()}")
print(f"Doublons sur question: {df.duplicated('question').sum()}")
print(f"Doublons sur question+answer: {df.duplicated(['question', 'answer']).sum()}")

In [ ]:
# Types
print("\nTypes:")
print(df.dtypes)


In [ ]:
# Longueur des textes
df['question_len'] = df['question'].str.len()
df['answer_len'] = df['answer'].str.len()

print("\nStatistiques longueur questions:")
print(df['question_len'].describe())
print("\nStatistiques longueur réponses:")
print(df['answer_len'].describe())

In [ ]:
print("Distribution par source:")
print(df['source'].value_counts())

In [ ]:
print("Top 20 focus_area:")
print(df['focus_area'].value_counts().head(20))

In [ ]:
# Visualisation de la distribution des sources
plt.figure(figsize=(10,6))
df['source'].value_counts().plot(kind='bar')
plt.title('Distribution par source')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Diagramme en Camembert des sources de données MedQuAD
plt.figure(figsize=(6,6))
df['source'].value_counts().plot(kind='pie', autopct='%1.1f%%')
plt.ylabel('')
plt.show()

In [ ]:
# Top 10 des focus du jeu de donnée
top10 = df['focus_area'].value_counts().head(10)
plt.figure(figsize=(8,4))
top10.plot(kind='barh')
plt.title('Top 10 focus area')
plt.xlabel("Frequency")
plt.tight_layout()
plt.show()

In [ ]:
# Visualisation longueur des questions
plt.figure(figsize=(10,6))
plt.hist(df['question_len'], bins=50, edgecolor='black')
plt.title('Distribution longueur des questions')
plt.xlabel('Longueur')
plt.ylabel('Fréquence')
plt.show()

In [ ]:
# Visualisation longueur des réponses
plt.figure(figsize=(10,6))
plt.hist(df['answer_len'], bins=50, edgecolor='black')
plt.title('Distribution longueur des réponses')
plt.xlabel('Longueur')
plt.ylabel('Fréquence')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10,4))
axes[0].hist(df['question_len'], bins=30)
axes[0].set_title('Questions')
axes[1].hist(df['answer_len'], bins=30)
axes[1].set_title('Réponses')
plt.show()

In [ ]:
# Boxplot comparatif
fig, axes = plt.subplots(1, 2, figsize=(12,5))
df.boxplot(column='question_len', ax=axes[0])
axes[0].set_title('Boxplot - Longueur questions')
df.boxplot(column='answer_len', ax=axes[1])
axes[1].set_title('Boxplot - Longueur réponses')
axes[1].set_yscale('log')
plt.tight_layout()
plt.show()

## **Constats Clés**

**Analyse/Observations**

- **Distribution des sources** : Déséquilibre marqué - GHR et GARD représentent ~66% des données. Risque de biais vers leurs pathologies couvertes.
- **Focus area** : Forte concentration sur cancers et maladies cardio-vasculaires. Nombreux sujets sous-représentés (apparition unique).
- **Qualité des données** : 1428 questions dupliquées (~9%), 48 doublons parfaits. Valeurs manquantes mineures (5 answer, 14 focus_area).
- **Longueur des textes** : Questions homogènes (~50 caractères). Réponses très hétérogènes (6 à 29000 caractères).

## **Décisions**

À partir des observations précédentes, plusieurs actions s'imposent pour préparer le dataset à la modélisation :

- **Nettoyer les doublons** : Éviter le sur-apprentissage et le biais sur les questions+réponses répétées
- **Gérer les valeurs manquantes** : Assurer l'intégrité des données d'entraînement
- **Traiter l'hétérogénéité des réponses** : Adapter aux contraintes de contexte des LLM
- **Équilibrer sources/focus_area** : Réduire les biais vers les catégories dominantes

In [ ]:
# Supprimer les doublons parfaits
df_clean = df.drop_duplicates()
print(f"Lignes après suppression doublons parfaits: {len(df_clean)}")

In [ ]:
# Compter occurrences par question
question_counts = df['question'].value_counts()
questions_frequentes = question_counts[question_counts > 1].index

print(f"Questions avec multiples réponses: {len(questions_frequentes)}")
print(f"Total de ces réponses multiples: {question_counts[questions_frequentes].sum()}")

In [ ]:
# Supprimer les réponses manquantes
df_clean = df_clean.dropna(subset=['answer'])
print(f"Après suppression answer manquants: {len(df_clean)}")

In [ ]:
# focus_area manquants
df_clean['focus_area'].isnull().sum()

In [ ]:
# Imputer focus_area manquants
df_clean['focus_area'] = df_clean['focus_area'].fillna('Other')
print(f"Focus_area manquants après imputation: {df_clean['focus_area'].isnull().sum()}")
print(f"Taille finale du dataset: {len(df_clean)}")

## **Risques de fuite (data leakage)**

Lors de la préparation des données, il faut éviter que des informations du futur (test) ne contaminent l'entraînement, ce qui donnerait des performances artificiellement élevées.

- **Split temporel** : Non applicable (pas de timestamp)
- **Doublons entre train/test** : Risque que la même question apparaisse dans les deux ensembles, faussant les performances. Solution : split par question unique, pas par ligne.
- **Preprocessing avant split** : Tout calcul (normalisation, imputation) doit être fit sur train seulement, puis transform sur test.

### **Aperću du `df_clean`**

In [ ]:
print(f"Shape finale: {df_clean.shape}")
print("\nColonnes:", df_clean.columns.tolist())

In [ ]:
print("Aperçu:")
df_clean.head()

In [ ]:
print("Valeurs manquantes:")
print(df_clean.isnull().sum())

In [ ]:
print("Distribution des sources:")
print(df_clean['source'].value_counts())

In [ ]:
df_clean['question_len'] = df_clean['question'].str.len()
df_clean['answer_len'] = df_clean['answer'].str.len()

print("Plage longueur réponses:")
print(f"Min: {df_clean['answer_len'].min()}, Max: {df_clean['answer_len'].max()}")
print(f"Mean: {df_clean['answer_len'].mean():.0f}, Median: {df_clean['answer_len'].median():.0f}")

In [ ]:
# Sauvegarde du dataset nettoyé
df_clean.to_csv('../data/medquad_clean.csv', index=False)
print("Dataset sauvegardé: ../data/medquad_clean.csv")